# Call our model server on Render

Our model server (`app.py`, on Render) serves the `/predict` API at
`https://hackathon-team1-weld-anomaly.onrender.com/gradio`.
This notebook calls it the way any program can: **plain HTTP in two steps**, the same as our Cloudflare app does.

1. `POST .../gradio_api/call/predict` with `{"data": ["<base64 image>"]}` → an `event_id` (a ticket)
2. `GET .../gradio_api/call/predict/<event_id>` → the result, after the line `event: complete`

Only Python's standard library is used, so it runs anywhere. (`call_server.py` does the same and also shows `gradio_client`.)

In [1]:
import base64, json, time, urllib.request
from pathlib import Path

URL = "https://hackathon-team1-weld-anomaly.onrender.com/gradio"

def get(url, timeout=90):
    with urllib.request.urlopen(url, timeout=timeout) as r:
        return r.status, r.read().decode()

def post_json(url, payload, timeout=90):
    req = urllib.request.Request(url, data=json.dumps(payload).encode(), headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read())

## 1 · Wake the server up

The free Render service sleeps after 15 minutes without visitors; the first request wakes it (about a minute).

In [2]:
start = time.time()
for attempt in range(12):
    try:
        status, _ = get(f"{URL}/gradio_api/info")
        if status == 200:
            print(f"server awake after {time.time() - start:.0f} s")
            break
    except Exception as err:
        print("still waking up:", type(err).__name__)
        time.sleep(10)

server awake after 53 s


## 2 · Send one X-ray tile per class

The tiles in `examples/` are RIAWELC test tiles. They stay **local only** (git-ignored): the dataset must not be re-uploaded. Only the numbers are printed here, not the images.

In [3]:
def predict(image_path):
    image_b64 = base64.b64encode(Path(image_path).read_bytes()).decode()
    # Step 1: send the input, get a ticket
    event_id = post_json(f"{URL}/gradio_api/call/predict", {"data": [image_b64]})["event_id"]
    # Step 2: fetch the result; it comes after the line "event: complete"
    _, text = get(f"{URL}/gradio_api/call/predict/{event_id}")
    lines = text.splitlines()
    data_line = lines[lines.index("event: complete") + 1]
    return json.loads(data_line.removeprefix("data:").strip())[0]

for name in ["no_defect", "porosity", "crack", "lack_of_penetration"]:
    t = time.time()
    r = predict(Path("examples") / f"{name}.png")
    print(f"{name:20s} risk {r['risk_level']:5s} | score {r['score']:.6f} | "
          f"{r['score_vs_threshold']:5.2f}x the threshold | anomaly {r['is_anomaly']!s:5s} | {time.time() - t:.1f} s")

no_defect            risk low   | score 0.000173 |  0.50x the threshold | anomaly False | 0.9 s


porosity             risk high  | score 0.011129 | 32.30x the threshold | anomaly True  | 0.7 s


crack                risk high  | score 0.004769 | 13.84x the threshold | anomaly True  | 0.9 s


lack_of_penetration  risk high  | score 0.010362 | 30.07x the threshold | anomaly True  | 0.9 s


## 3 · What the answer contains

In [4]:
r = predict(Path("examples") / "crack.png")
for key, value in r.items():
    shown = f"<PNG image, {len(value)} base64 characters>" if key.endswith("_png_base64") else value
    print(f"{key:26s} {shown}")

score                      0.004769
risk_level                 high
is_anomaly                 True
score_vs_threshold         13.84
threshold                  0.000345
t_low                      0.000255
t_high                     0.001442
reconstruction_png_base64  <PNG image, 5612 base64 characters>
error_map_png_base64       <PNG image, 17256 base64 characters>


**Reading it:** `score` is the mean reconstruction error (MSE). Above `t_low` the tile is **check**, above `t_high` it is **high**. `is_anomaly` compares with the 95% threshold. The two images are the model's redraw and the error heat map that the app shows.